# 10 — Rhino STL Generation

**Author:** Héctor Fernández Pinacho  
**Lab:** IDEAL Lab  
**Purpose:** Generate STL mesh files for every propeller configuration using RhinoCompute and the `propeller.gh` Grasshopper definition.

This notebook is a self-contained geometry-export step for the IDEAL Lab propeller pipeline. It reads the geometrical parameter table produced by notebook 1, submits each configuration to RhinoCompute via the `compute-rhino3d` Python client, decodes the returned mesh, writes it as a binary STL, and exports a results CSV.

```text
csv/prop_geometrical_params.csv  →  RhinoCompute  →  stl/prop_<id>.stl
                                                   →  csv/rhino_stl_generation.csv
```

It does not generate LHS samples, mass estimates, NACA codes, or QPROP inputs.

---
## How the pipeline works

The original propeller configurator website uses a Node.js app server to bridge the browser and RhinoCompute. This notebook replicates the same pipeline entirely in Python:

1. A **local HTTP server** is started in a background thread to serve `propeller.gh` — exactly as the Node app server does via its `/definition/<md5>` endpoint. RhinoCompute fetches and caches the definition from this URL.
2. Each row of `prop_geometrical_params.csv` is mapped to the camelCase parameter names that the Grasshopper definition expects.
3. The `exportSolution` flag is set to `True`, which tells Grasshopper to generate the high-quality `MeshFinal` output (the same flag the website sets when the user clicks Export).
4. `compute_rhino3d.Grasshopper.evaluateDefinition` sends the definition URL and the DataTree inputs to RhinoCompute and returns a JSON response.
5. The `MeshFinal` output is decoded from the response, converted to a binary STL, and saved.


## 1. Imports

In [24]:
# Imports

import hashlib
import json
import struct
import threading
import time
from http.server import HTTPServer, SimpleHTTPRequestHandler
from pathlib import Path

import numpy as np
import pandas as pd
import rhino3dm
from compute_rhino3d import Grasshopper, Util
from tqdm.auto import tqdm

## 2. Constants and paths

All tunable settings live here. Under normal use, only this section needs editing.

### RhinoCompute URL
Set `RHINO_COMPUTE_URL` to wherever your RhinoCompute instance is running. The default is the local port used by the Rhino 7 compute package (`http://localhost:6500/`). If you are running the standalone `compute.geometry` server it typically uses `http://localhost:8081/`.

### File server port
`GH_SERVER_PORT` is the local port used to serve `propeller.gh` to RhinoCompute. Any free port works.

In [25]:
# Configuration

# =============================================================================
# 1) RHINOCOMPUTE
# =============================================================================

RHINO_COMPUTE_URL = "http://localhost:5000/"   # adjust to your RhinoCompute URL
RHINO_COMPUTE_API_KEY = ""                     # leave empty if no key is required


# =============================================================================
# 2) LOCAL FILE SERVER (serves propeller.gh to RhinoCompute)
# =============================================================================

GH_SERVER_PORT = 9876                          # any free local port


# =============================================================================
# 3) GRASSHOPPER PARAMETERS
# =============================================================================

# This flag triggers MeshFinal generation inside the .gh definition.
# It mirrors the exportSolution=True flag sent by the website on export.
EXPORT_SOLUTION = True

# Fixed positional offset passed to the definition (same value the website uses).
POSITION_OFFSET = 50

# Name of the Grasshopper output parameter that contains the final mesh.
GH_OUTPUT_MESH_PARAM = "MeshFinal"


# =============================================================================
# 4) PATHS
# =============================================================================

BASE_DIR   = Path.cwd()
CSV_DIR    = BASE_DIR / "csv"
STL_DIR    = BASE_DIR / "stl"
STEP_DIR   = BASE_DIR / "step"

GH_FILE    = BASE_DIR / "propeller.gh"                              # Grasshopper definition
PARAMS_CSV = CSV_DIR  / "representative_geometry.csv"               # input: geometry table
RESULTS_CSV= CSV_DIR  / "rhino_stl_generation_representative.csv"   # output: results table


# =============================================================================
# 5) BATCH SETTINGS
# =============================================================================

# Set to None to process every row, or an integer to process a subset (useful for testing).
MAX_CONFIGS = None

# Seconds to wait between RhinoCompute calls. Increase if the server overloads.
INTER_CALL_DELAY_S = 0.0

## 3. Input validation

The notebook checks that all required files and directories exist before starting the batch.

In [26]:
# Input validation

if not GH_FILE.exists():
    raise FileNotFoundError(
        f"Grasshopper definition not found: {GH_FILE}\n"
        "Copy propeller.gh from the website source (02_Relevant_Files/files/) to the dataset root."
    )

if not PARAMS_CSV.exists():
    raise FileNotFoundError(
        f"Geometrical parameters CSV not found: {PARAMS_CSV}\n"
        "Run notebook 1 first."
    )

STL_DIR.mkdir(parents=True, exist_ok=True)
STEP_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

params_df = pd.read_csv(PARAMS_CSV)

if MAX_CONFIGS is not None:
    params_df = params_df.head(MAX_CONFIGS)

print(f"Grasshopper definition : {GH_FILE.name}  (MD5: {hashlib.md5(GH_FILE.read_bytes()).hexdigest()})")
print(f"Configurations to process: {len(params_df)}")
print(f"STL output directory   : {STL_DIR}")
print(f"RhinoCompute URL       : {RHINO_COMPUTE_URL}")

Grasshopper definition : propeller.gh  (MD5: d0fd3634cb0a428e36972da5585cdf4c)
Configurations to process: 100
STL output directory   : c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\stl
RhinoCompute URL       : http://localhost:5000/


## 4. Start local file server

RhinoCompute fetches the Grasshopper definition from a URL the first time it is used, then caches it. A minimal `HTTPServer` running in a background thread serves `propeller.gh` from the dataset root directory.

The URL format is:
```
http://localhost:<GH_SERVER_PORT>/propeller.gh
```

This is equivalent to the `/definition/<md5>` endpoint in the Node app server.

In [27]:
# Start local HTTP server to serve propeller.gh

class _QuietHandler(SimpleHTTPRequestHandler):
    """SimpleHTTPRequestHandler with logging suppressed."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, directory=str(BASE_DIR), **kwargs)

    def log_message(self, format, *args):  # noqa: A002
        pass  # suppress per-request stdout noise


_server = HTTPServer(("localhost", GH_SERVER_PORT), _QuietHandler)
_server_thread = threading.Thread(target=_server.serve_forever, daemon=True)
_server_thread.start()

GH_URL = f"http://localhost:{GH_SERVER_PORT}/{GH_FILE.name}"
print(f"File server started  : {GH_URL}")
print("(Runs in background — it shuts down automatically when the kernel stops.)")

File server started  : http://localhost:9876/propeller.gh
(Runs in background — it shuts down automatically when the kernel stops.)


## 5. Helper functions

These functions encapsulate the three steps of each RhinoCompute call:

1. **`row_to_gh_inputs`** — maps the CSV column names (with spaces) to the camelCase names the Grasshopper definition expects, and adds the two fixed parameters `positionOffset` and `exportSolution`.
2. **`call_rhinocompute`** — builds the `DataTree` inputs and calls `Grasshopper.evaluateDefinition`.
3. **`decode_mesh_final`** — walks the response JSON, finds the `MeshFinal` output, and returns a `rhino3dm.Mesh` object.
4. **`mesh_to_stl`** — serialises a `rhino3dm.Mesh` to a binary STL file.

In [28]:
# Helper functions

# ---------------------------------------------------------------------------
# Column name mapping: CSV header → Grasshopper input parameter name
# ---------------------------------------------------------------------------
# The CSV uses the same labels as the website sliders but with spaces and
# lowercase words.  The Grasshopper definition expects camelCase names that
# mirror the JavaScript SLIDER_CONFIG keys in script.js.

CSV_TO_GH = {
    "radius":          "impellerRadius",
    "ring height":     "impellerHeight",
    "ring thickness":  "impellerThickness",
    "blade count":     "bladeCount",
    "inner thickness": "innerThickness",
    "inner max pos":   "innerMaxPos",
    "inner camber":    "innerCamber",
    "inner chord":     "innerChord",
    "inner angle":     "innerAngle",
    "mid radial pos":  "middlePos",
    "mid chord":       "middleChord",
    "mid angle":       "middleAngle",
    "outer thickness": "outerThickness",
    "outer max pos":   "outerMaxPos",
    "outer camber":    "outerCamber",
    "outer chord":     "outerChord",
    "outer angle":     "outerAngle",
}


def row_to_gh_inputs(row: pd.Series) -> dict:
    """Convert a CSV row to the dict of GH input names and values."""
    inputs = {gh_name: float(row[csv_col]) for csv_col, gh_name in CSV_TO_GH.items()}
    inputs["positionOffset"] = int(POSITION_OFFSET)
    inputs["exportSolution"] = bool(EXPORT_SOLUTION)
    return inputs


def call_rhinocompute(gh_url: str, inputs: dict) -> dict:
    """
    Send inputs to RhinoCompute and return the parsed JSON response.

    Each key-value pair in `inputs` becomes a Grasshopper DataTree with a
    single branch at path {0} — identical to what the Node app server does
    in routes/solve.js.
    """
    trees = []
    for name, value in inputs.items():
        tree = Grasshopper.DataTree(name)
        tree.Append([0], [value])
        trees.append(tree)

    return Grasshopper.EvaluateDefinition(gh_url, trees)


def decode_mesh_final(response_json: dict, output_name: str = GH_OUTPUT_MESH_PARAM):
    """
    Extract and decode the mesh from the RhinoCompute response.

    Returns a tuple (mesh, errors, all_output_names) where:
      - mesh is a rhino3dm.Mesh or None if not found / not decodable
      - errors is a list of Grasshopper error strings (empty if none)
      - all_output_names is the list of output ParamNames in the response
    """
    errors = [e.get("Message", str(e)) for e in response_json.get("errors", [])]
    values = response_json.get("values", [])
    all_output_names = [v["ParamName"] for v in values]

    # Find the target output
    target = next((v for v in values if v["ParamName"] == output_name), None)
    if target is None:
        return None, errors, all_output_names

    # Walk the InnerTree and decode the first valid mesh object
    for branch in target.get("InnerTree", {}).values():
        for item in branch:
            raw = item.get("data")
            if raw is None:
                continue
            try:
                # Try Draco-compressed base64 first (starts with "RFJBQ08" = "DRACO")
                s = raw.strip('"') if isinstance(raw, str) else ""
                if s.startswith("RFJBQ08") or s.startswith("DRACO"):
                    obj = rhino3dm.DracoCompression.DecompressBase64String(s)
                    if isinstance(obj, rhino3dm.Mesh):
                        return obj, errors, all_output_names
                # Fall back to OpenNURBS JSON encoding
                obj_data = json.loads(raw) if isinstance(raw, str) else raw
                obj = rhino3dm.CommonObject.Decode(obj_data)
                if isinstance(obj, rhino3dm.Mesh):
                    return obj, errors, all_output_names
            except Exception:
                continue

    return None, errors, all_output_names


def mesh_to_stl(mesh: rhino3dm.Mesh, filepath: Path) -> dict:
    """
    Write a rhino3dm.Mesh to a binary STL file.

    Quads are split into two triangles each before writing.
    Returns a dict with volume_mm3, n_vertices, and n_triangles.
    """
    mesh.Faces.ConvertQuadsToTriangles()

    vertices = mesh.Vertices
    faces    = mesh.Faces

    n_triangles = len(faces)
    n_vertices  = len(vertices)

    with filepath.open("wb") as f:
        f.write(b"\x00" * 80)                        # 80-byte header
        f.write(struct.pack("<I", n_triangles))      # triangle count

        for i in range(n_triangles):
            face = faces[i]
            v0 = vertices[face[0]]
            v1 = vertices[face[1]]
            v2 = vertices[face[2]]

            # Compute face normal
            ax, ay, az = v1.X - v0.X, v1.Y - v0.Y, v1.Z - v0.Z
            bx, by, bz = v2.X - v0.X, v2.Y - v0.Y, v2.Z - v0.Z
            nx = ay * bz - az * by
            ny = az * bx - ax * bz
            nz = ax * by - ay * bx
            length = (nx**2 + ny**2 + nz**2) ** 0.5
            if length > 0:
                nx, ny, nz = nx / length, ny / length, nz / length

            f.write(struct.pack("<fff", nx, ny, nz))
            for v in (v0, v1, v2):
                f.write(struct.pack("<fff", v.X, v.Y, v.Z))
            f.write(struct.pack("<H", 0))            # attribute byte count

    # Approximate volume via divergence theorem (signed, for diagnostics)
    volume = 0.0
    for i in range(n_triangles):
        face = faces[i]
        v0 = vertices[face[0]]
        v1 = vertices[face[1]]
        v2 = vertices[face[2]]
        volume += (
            v0.X * (v1.Y * v2.Z - v2.Y * v1.Z)
            + v1.X * (v2.Y * v0.Z - v0.Y * v2.Z)
            + v2.X * (v0.Y * v1.Z - v1.Y * v0.Z)
        ) / 6.0

    return {
        "volume_mm3":   abs(volume),
        "n_vertices":   n_vertices,
        "n_triangles":  n_triangles,
    }

def mesh_to_step(mesh: rhino3dm.Mesh, filepath: Path) -> bool:
    """
    Export a rhino3dm.Mesh to a STEP file by running a Python script on the
    RhinoCompute server via util.PythonEvaluate.

    The server-side script receives the mesh encoded as a rhino3dm JSON object,
    converts it to a Brep, and exports it to STEP using Rhino.FileIO.
    Returns True on success, False on failure.
    """
    script = """
import Rhino
import System

mesh_obj = Rhino.Geometry.Mesh()
for pt in input["vertices"]:
    mesh_obj.Vertices.Add(pt[0], pt[1], pt[2])
for f in input["faces"]:
    mesh_obj.Faces.AddFace(f[0], f[1], f[2])
mesh_obj.Normals.ComputeNormals()
mesh_obj.Compact()

breps = Rhino.Geometry.Brep.CreateFromMesh(mesh_obj, True)
if breps is None or len(breps) == 0:
    output["ok"] = False
    output["error"] = "Brep.CreateFromMesh returned nothing"
else:
    brep = breps[0]
    tmp = System.IO.Path.GetTempFileName() + ".stp"
    write_opts = Rhino.FileIO.FileWriteOptions()
    write_opts.WriteSelectedObjectsOnly = False
    doc = Rhino.RhinoDoc.CreateHeadless(None)
    doc.Objects.AddBrep(brep)
    ok = Rhino.FileIO.FileStep.Write(tmp, doc, write_opts)
    if ok:
        with open(tmp, "rb") as fh:
            import base64
            output["data"] = base64.b64encode(fh.read()).decode()
        output["ok"] = True
    else:
        output["ok"] = False
        output["error"] = "FileStep.Write returned False"
    doc.Dispose()
"""
    # Collect mesh data to send to the server
    verts = [[mesh.Vertices[i].X, mesh.Vertices[i].Y, mesh.Vertices[i].Z]
             for i in range(len(mesh.Vertices))]
    faces = [[mesh.Faces[i][0], mesh.Faces[i][1], mesh.Faces[i][2]]
             for i in range(len(mesh.Faces))]

    result = Util.PythonEvaluate(script, {"vertices": verts, "faces": faces}, ["ok", "data", "error"])

    if result.get("ok"):
        import base64
        filepath.write_bytes(base64.b64decode(result["data"]))
        return True
    else:
        raise RuntimeError(result.get("error", "Unknown error from RhinoCompute PythonEvaluate"))

## 6. Configure RhinoCompute client

In [29]:
# Configure compute_rhino3d client

Util.url    = RHINO_COMPUTE_URL
Util.apiKey = RHINO_COMPUTE_API_KEY

print(f"compute_rhino3d configured → {RHINO_COMPUTE_URL}")

compute_rhino3d configured → http://localhost:5000/


## 7. Batch STL generation

For every row in `prop_geometrical_params.csv` the notebook:

1. Maps the row to Grasshopper input names.
2. Calls RhinoCompute.
3. Decodes the `MeshFinal` output.
4. Writes the STL to `stl/prop_<config_id>.stl`.
5. Records the outcome in a results list.

Failed configurations are recorded with `geometry_ok = False` and a descriptive `error_message`. The batch continues regardless of individual failures.

In [30]:
# DEBUG — test decode of MeshFinal
import rhino3dm, json as _json

test_row = params_df.iloc[0]
test_inputs = row_to_gh_inputs(test_row)

trees = []
for name, value in test_inputs.items():
    tree = Grasshopper.DataTree(name)
    tree.Append([0], [value])
    trees.append(tree)

raw = Grasshopper.EvaluateDefinition(GH_URL, trees)

# Get MeshFinal data
for v in raw.get("values", []):
    if v["ParamName"] == "MeshFinal":
        for branch, items in v.get("InnerTree", {}).items():
            for item in items:
                data = item.get("data")
                s = data.strip('"') if isinstance(data, str) else ""
                print(f"data starts with: {s[:20]!r}")
                print(f"is Draco: {s.startswith('RFJBQ08')}")
                
                # Try Draco decode
                try:
                    obj = rhino3dm.DracoCompression.DecompressBase64String(s)
                    print(f"Draco decode result: {type(obj)}")
                    if obj:
                        print(f"  Is Mesh: {isinstance(obj, rhino3dm.Mesh)}")
                        if isinstance(obj, rhino3dm.Mesh):
                            print(f"  Vertices: {len(obj.Vertices)}")
                            print(f"  Faces: {len(obj.Faces)}")
                except Exception as e:
                    print(f"Draco decode error: {e}")
                
                # Try CommonObject decode
                try:
                    obj2 = rhino3dm.CommonObject.Decode(_json.loads(data) if isinstance(data, str) else data)
                    print(f"CommonObject decode result: {type(obj2)}")
                except Exception as e:
                    print(f"CommonObject decode error: {e}")


data starts with: 'RFJBQ08CAgEBAAAA560B'
is Draco: True
Draco decode result: <class 'rhino3dm._rhino3dm.Mesh'>
  Is Mesh: True
  Vertices: 22247
  Faces: 38414
CommonObject decode error: Decode(): incompatible function arguments. The following argument types are supported:
    1. (jsonObject: dict) -> rhino3dm._rhino3dm.CommonObject

Invoked with: '...'


In [31]:
# Batch STL generation

results = []

for _, row in tqdm(params_df.iterrows(), total=len(params_df), desc="Generating STLs"):
    config_id = int(row["config_id"])
    stl_path  = STL_DIR / f"prop_{config_id}.stl"
    step_path = STEP_DIR / f"prop_{config_id}.step"

    record = {
        "config_id":           config_id,
        "geometry_ok":         False,
        "stl_path":            str(stl_path.relative_to(BASE_DIR)),
        "volume_mm3":          None,
        "volume_cm3":          None,
        "volume_m3":           None,
        "n_vertices":          None,
        "n_triangles":         None,
        "selected_output_name": GH_OUTPUT_MESH_PARAM,
        "error_message":       "",
        "step_ok":             False,
        "step_path":           str(step_path.relative_to(BASE_DIR)),
        "step_error":          "",
    }

    try:
        inputs       = row_to_gh_inputs(row)
        response     = call_rhinocompute(GH_URL, inputs)
        mesh, errors, output_names = decode_mesh_final(response)

        if mesh is None:
            non_empty = [
                v["ParamName"]
                for v in response.get("values", [])
                if any(branch for branch in v.get("InnerTree", {}).values())
            ]
            record["error_message"] = (
                f"No decodable mesh objects found in output '{GH_OUTPUT_MESH_PARAM}'. "
                f"Non-empty outputs: {non_empty}. "
                f"Grasshopper errors: {errors}."
            )
        else:
            stats = mesh_to_stl(mesh, stl_path)
            try:
                step_ok = mesh_to_step(mesh, step_path)
                record["step_ok"] = step_ok
            except Exception as step_exc:
                record["step_error"] = str(step_exc)
            record.update({
                "geometry_ok": True,
                "volume_mm3":  round(stats["volume_mm3"], 2),
                "volume_cm3":  round(stats["volume_mm3"] / 1e3, 4),
                "volume_m3":   round(stats["volume_mm3"] / 1e9, 9),
                "n_vertices":  stats["n_vertices"],
                "n_triangles": stats["n_triangles"],
            })

    except Exception as exc:
        record["error_message"] = str(exc)

    results.append(record)

    if INTER_CALL_DELAY_S > 0:
        time.sleep(INTER_CALL_DELAY_S)

results_df = pd.DataFrame(results)

n_ok   = results_df["geometry_ok"].sum()
n_fail = len(results_df) - n_ok
print(f"\nDone — {n_ok} succeeded, {n_fail} failed.")

Generating STLs:   2%|▏         | 2/100 [00:42<35:06, 21.49s/it]


KeyboardInterrupt: 

## 8. Export results CSV

In [ ]:
# Export results CSV

results_df.to_csv(RESULTS_CSV, index=False)
print(f"Results written to: {RESULTS_CSV}")
results_df.head(10)

## 9. Summary

In [ ]:
# Summary

ok_df   = results_df[results_df["geometry_ok"]]
fail_df = results_df[~results_df["geometry_ok"]]

print("=" * 60)
print(f"Total configurations : {len(results_df)}")
print(f"  Successful STLs    : {len(ok_df)}")
print(f"  Failed             : {len(fail_df)}")

if len(ok_df) > 0:
    print()
    print("Volume statistics (successful configs, mm³):")
    print(f"  Mean    : {ok_df['volume_mm3'].mean():,.1f}")
    print(f"  Median  : {ok_df['volume_mm3'].median():,.1f}")
    print(f"  Min     : {ok_df['volume_mm3'].min():,.1f}")
    print(f"  Max     : {ok_df['volume_mm3'].max():,.1f}")
    print()
    print("Mesh statistics (successful configs):")
    print(f"  Avg triangles : {ok_df['n_triangles'].mean():,.0f}")
    print(f"  Avg vertices  : {ok_df['n_vertices'].mean():,.0f}")

if len(fail_df) > 0:
    print()
    print(f"First {min(5, len(fail_df))} failed config IDs:")
    for _, r in fail_df.head(5).iterrows():
        print(f"  config_id={int(r['config_id'])}: {r['error_message'][:120]}")

print("=" * 60)